# The Anatomy of a Power Outage

The U.S. Department of Energy (DOE) provides critical information about the status and impacts of energy sector disruptions through the **Environment for Analysis of Geo-Located Energy Information (EAGLE-I)** system, operated by Oak Ridge National Laboratory. EAGLE-I supports monitoring of energy infrastructure assets, reporting of power outages, visualization of threats to energy infrastructure, and coordination of emergency response and recovery efforts.

Effectively responding to and restoring power during disasters depends on having timely, accurate, and actionable data. In this project, your goal is to help develop tools and processes to better understand the **characteristics and causes of power outages** in the United States.

---

## Project Overview

This project has four parts. The first two are completed in this notebook; the third is in a separate Part3Weather.ipynb Jupyter notebook.

### Part 1: Clean the Power Outage Data Using MPI

- Use an `mpi4py` script to clean the raw EAGLE-I data.
- Evaluate how the script performs when dividing the workload among different numbers of processors.
- Analyze how well the script **scales** across the processors available on a single node.

**So what?**  
Understanding how performance scales with parallelism is a **core concept in High Performance Computing (HPC)**.

### Part 2: Analyze Power Outage Patterns

- Explore the cleaned outage dataset using **histograms** and **K-means clustering** or 
- Group outages based on the **region** and **month** they occur in.
- Identify when and where outages are most likely to happen in different parts of the U.S.

**So what?**  
Machine learning and data visualization are **powerful tools for understanding patterns in large datasets**.

### Part 3: Explore NOAA Severe Weather Events (in `Weather.ipynb`)

- Apply the same analysis (histograms and K-means clustering) to **NOAA's severe weather events** database.
- Cluster and characterize events by **region** and **month**.
- Understand when and where **severe weather** is most common.
**So what?** Once you learn a method you can apply it to more than one kind of data. 
### Part 4: Final Synthesis

- Reflect on the results from your MPI performance analysis. What factors influence how efficiently work can be parallelized on a node?
- Use your plots and cluster results from Parts 2 and 3 to:
  - Characterize **power outage** and **weather event** patterns.
  - **Discuss whether and how the two are related**.

# Big Data Cleaning and Processing with MPI

You will use a script called `GenerateOutageCSV.py` to clean and process the EAGLE-I data. The script is set up to use `mpi4py` to process the data in parallel. You will not only process the data, but also explore how the execution of the script scales as you run it on more processors. 

> *Scaling on processors refers to how the performance (e.g., execution time) of a program changes as more processors are used to run it.*

## A Note About the Raw Data

The raw Eagle-I data is structured so that each row represents a 15-minute increment for each county, and gives the number of customers who were without power during that time period in the `"sum"` column. This dataset spans from 2016 to 2022. 

As you can imagine, with the number of 15-minute increments in a year and the number of counties in the U.S., there are a lot of rows. This is a classic *time series* dataset, and its large size is why we need MPI to help process it.

Note that "customers" does **not** mean people—one customer could be a single house, an apartment complex, or a business. 

## What Are We Doing?

We are going to process the raw data into a format that will give us:

- The **length of an outage**, defined as the time interval during which the number of customers out rose above 10%, and the latest time after which it fell back below 10%.
- The `"sum"` in the processed dataset will represent the **average number of customers without power during that outage interval**.

---

## What Comes Next

The next section does two things:

1. Explains what is in the script.
2. Explains how to do a scaling study with the script.

---

## Deliverable

The **first part** of your deliverables for this project will be to **present the scaling study**.

# Imports
First you will import all the Python packages you need for this project below.

*Python packages are collections of modules that provide reusable code for specific tasks, such as data analysis or parallel computing. You import them using the `import` statement so you can access the tools and functions they contain in your script.*

To run the cell below, click on the cell and then type `Shift` and `Enter`  
or `Return` at the same time. This is how you will run all the code cells inside a Jupyter Notebook.

In [ ]:
import os
import sys
import math
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global Variables

## Fips Codes and Census Regions

The data provided in the EAGLE-I datasets includes FIPS codes for outages reported from different areas. FIPS codes are unique identifiers that describe a specific geographic location. The FIPS codes provided in the EAGLE-I dataset are county-level; which means they atr 5 total digits. The first two digits provide the state and the last 3 determine the county within that state.

More regarding FIPS codes: https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt

In the python script ```GenerateOutageCSV.py```, we provide a dictionary that keys on the two digit FIPS code for states in order to identify the state of each outage. We also provide a numerical identifier for which census region of the US each outage occurs in. We do this to have additional features available to use in our unsupervised learning later on. The dictionary provided in the script and a map displaying the US census regions are displayed below for viewing convienience.

![Map with Census Regions Indicated](./Images/census_regions.gif)

```python
# Regions 
#   1: Pacific
#   2: Mountain
#   3: West North Central
#   4: West South Central
#   5: East North Central
#   6: East South Central
#   7: New England
#   8: Mid-Atlantic
#   9: South Atlantic
FIPS_TO_STATE = {
    '01': ('Alabama', 6),
    '02': ('Alaska', 1),
    '04': ('Arizona', 2),
    '05': ('Arkansas', 4),
    '06': ('California', 1),
    '08': ('Colorado', 2),
    '09': ('Connecticut', 7),
    '10': ('Delaware', 9),
    '11': ('District of Columbia', 9),
    '12': ('Florida', 9),
    '13': ('Georgia', 9),
    '15': ('Hawaii', 1),
    '16': ('Idaho', 2),
    '17': ('Illinois', 5),
    '18': ('Indiana', 5),
    '19': ('Iowa', 3),
    '20': ('Kansas', 3),
    '21': ('Kentucky', 6),
    '22': ('Louisiana', 4),
    '23': ('Maine', 7),
    '24': ('Maryland', 9),
    '25': ('Massachusetts', 7),
    '26': ('Michigan', 5),
    '27': ('Minnesota', 3),
    '28': ('Mississippi', 6),
    '29': ('Missouri', 3),
    '30': ('Montana', 2),
    '31': ('Nebraska', 3),
    '32': ('Nevada', 2),
    '33': ('New Hampshire', 7),
    '34': ('New Jersey', 8),
    '35': ('New Mexico', 2),
    '36': ('New York', 8),
    '37': ('North Carolina', 9),
    '38': ('North Dakota', 3),
    '39': ('Ohio', 5),
    '40': ('Oklahoma', 4),
    '41': ('Oregon', 1),
    '42': ('Pennsylvania', 8),
    '44': ('Rhode Island', 7),
    '45': ('South Carolina', 9),
    '46': ('South Dakota', 3),
    '47': ('Tennessee', 6),
    '48': ('Texas', 4),
    '49': ('Utah', 2),
    '50': ('Vermont', 7),
    '51': ('Virginia', 9),
    '53': ('Washington', 1),
    '54': ('West Virginia', 9),
    '55': ('Wisconsin', 5),
    '56': ('Wyoming', 2),
}
```

# MPI (mpi4py)

## Message Passing Interface (MPI) Exercises

High-Performance Computing is a field that leverages parallel processing to solve complex problems efficiently. 

A popular approach to parallel programming is the `Message Passing Interface (MPI)`.

In this tutorial, we will use Python and MPI to analyze power outage data from EAGLE-i, which contains information about the number of power outages, aggregated by county, in 15-minute time intervals. 

We will explore how using multiple processors to our advantage can save time when working with large datasets, and use the MPI for Python (`mpi4py`) package, which provides a simple Python interface to the MPI.

MPI is designed to allows users to easily perform distributed parallel processing across multiple processors on a single computer or even multiple nodes on an HPC system.

### Pre-requisites

- Make sure the Eagle-i data is in the `eaglei_outages` folder. You can do that by navigating to  `/global/cfs/cdirs/m4388/Project3/Data` in the  panel on the left  or by activating the `ls` in the cell below. 

(The 2014-2022 dataset is also available at: https://drive.google.com/drive/u/0/folders/15skV7CWnMBLkrGIPmLrh1GgNPA00iUuB)


In [ ]:
!ls /global/cfs/cdirs/m4388/Project3/Data/eaglei_outages


### Getting Started
Below is an overview of the project outline as whole for this half of the project. 
1. Load and pre-process the power outage data
2. Divide the data into chunks to distribute among MPI processes
3. Implement parallel processing using MPI
In the 2nd half of the proejct you will Analyze and interpret the results of the data. 

### Additional Resources: 
* Commonly Used HPC Terms: https://researchcomputing.princeton.edu/learn/glossary
* MPI Learning Resources: https://researchcomputing.princeton.edu/education/external-online-resources/mpi
  * MPI Tutorial by LBNL: https://hpc-tutorials.llnl.gov/mpi/
* Additional Research Computing Resources: https://researchcomputing.princeton.edu/learn/tutorials/external-resources-learning

## Using Conda Environments
We will use a Conda environment with mpi4py and other packages we need installed in order to run the scripts we are using today. A Conda environment is an isolated workspace that contains a specific collection of Python packages and dependencies. It allows you to manage software versions separately from your system or other projects to avoid conflicts.


The simplest way to clone an environment to another locations is by executing the following command on the command line:

```bash
conda create --clone <environment_to_clone> --name <new_environment_name> --prefix <path_to_new_environment>
```
If the environment you're cloning is in the default Conda environment folder you can simply clone using the name of the environment. If it's in another location, you will need to specify the full path. (The same is true when activating environments.

**Note:** You do **not** need to clone or create a new environment right now. An environment is already available on the systems we are using. 

### Activiating the existing conda env. 
To use a Conda environment, you must activate it. Activating a Conda environment sets up your terminal or notebook to use the specific Python interpreter and packages from that environment, ensuring your code runs with the correct dependencies.

You can activate it with:
```bash
conda activate </path/to/environment/environment_name>
```
For this camp we will activate the `nersc-python` conda env. that contains the mpi4py in the batch script with these lines:
```
module load conda
conda activate /global/common/software/nersc/pe/conda-envs/24.1.0/python-3.11/nersc-python

```

When your environment is activated, you can check what packages are installed and install (most) new packages with:
```bash
# check installed packages
conda list

# install new package
conda install <package_name>
```

When you are done in your environment, you can exit it via:
```bash
conda deactivate
```

### Practice Exercise

The environment will be activated automatically inside the script when we launch the job, but here you’ll activate it manually so you can see the process. This exercise isn’t required for your project, but it’s a useful learning step if you haven’t worked with Python environments before.

Steps:

- In JupyterHub, click the + icon at the top of this notebook to open the Launcher.
- Scroll down to the "Other" section and click Terminal to open a terminal window.
- In the terminal, follow these steps:
- Load the Conda module
- Activate the nersc-python environment
- List the Python packages installed in that environment
- Deactivate the environment when finished

We have provided a directory called my_env should you choose to build any custom env for your project. 

## GenerateOutageCSV.py Script
We will be using the ```GenerateOutageCSV.py``` script to mold the EAGLE-I data into a format that will be more ideal for our analysis in the notebook.

### 1.  Load and pre-process the power outage data
We'll first start by loading in the power outage data sets. The code for that is below.

```python
import pandas as pd

def readEagle(years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021], drop_columns = ['county', 'state']): 
    cleaned_eaglei = pd.DataFrame()
    for year in years:
        eagle_csv = pd.read_csv(f'eaglei_outages/eaglei_outages_{year}.csv')
        cleaned_eaglei = pd.concat([cleaned_eaglei, eagle_csv], ignore_index = True)
        cleaned_eaglei.drop(drop_columns, axis=1, inplace=True)    # Leaves fips, sum, run_start_time
    
    return cleaned_eaglei
```

### 2. Divide the data into chunks to distribute among MPI processes

As you may have noticed, we are working with a lot of data. In order to speed up the process of data pre-processing, let's utilize MPI and multiple processors. Below is the function we use to split the data into chunks for each MPI rank. We split the data based on FIPS codes and the total number of ranks. This way each rank should be looking at the data for a subset of FIPS codes.

```python
# Function to determine the range of fips code my rank should work with
def fipsRange(df, Rank, Size):
    # Create a list of each unique fips code
    fips = list(df['fips_code'].unique())

    # Determine indices for fips code selection by rank
    WorkLoadPerProc = len(fips) // Size
    Remainder = len(fips) % Size
    StartIndx = Rank * WorkLoadPerProc + min(Rank, Remainder)
    EndIndx = StartIndx + WorkLoadPerProc + (1 if Rank < Remainder else 0)

    return StartIndx, EndIndx, fips
```

### 3. Implement parallel processing using MPI

Below is the primary code which initializes our MPI and calls the functions we showed above. We use the ```bcast``` MPI function to send the initial data from rank 0 to all other ranks. Then we utilize our ```Outage``` function (which calls ```fipsRange```) to split the fips codes amongst each rank. Finally, after each rank does its part of the work, we use the ```gather``` MPI function to gather each section of the now modified data back to rank 0.

```python
def main(args):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()

    if rank == 0:
        # Record the start time for time tracking
        start_time = MPI.Wtime() 
        years = []

        # Argument handling
        if len(args) < 2:
            print(f"error: {args[0]} [<years>] <outage_length>")
            return 1
        if len(args) >= 3:
            start_year=args[1]
            end_year=args[len(args)-2]
            for i in range(1, len(args)-1):
                years.append(args[i])
        if years:
            cleaned_eaglei = readEagle(years = years)
        else:
            cleaned_eaglei = readEagle()
    else:
        cleaned_eaglei = pd.DataFrame()

    # Broadcast the initial data to all ranks
    cleaned_eaglei = comm.bcast(cleaned_eaglei, root=0) 

    # Utilize MPI to split the data amongst the ranks and perform our processing
    agg_outages_per_proc = Outage(cleaned_eaglei, rank, size, float(args[len(args)-1]))

    # Gather all the modified data back to rank 0
    aggregated_Outages = comm.gather(agg_outages_per_proc, root=0)

    # Record data in a CSV and report total MPI time
    if(rank == 0):
    
        end_time = MPI.Wtime()
        
        aggOutagesDF = pd.concat(aggregated_Outages)
    
        aggOutagesDF.sort_values(by=['State', 'OutageStart'], inplace=True)
    
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        aggOutagesDF.to_csv(f'OutageCSVs/{start_year}-{end_year}-Outages.csv', index=False, header=True)
        print(f"Total time: {end_time - start_time} seconds")
```

## Bash Scripts

To run our Python script on Perlmutter, we will utilize a batch script to submit the job to the queue. Our batch script can be seen in ```submit_python_outage.sbatch```. Here is an example configuratrion:

```bash
#!/bin/bash
#SBATCH --constraint=cpu
#SBATCH --nodes=1
#SBATCH --time=10
#SBATCH --ntasks-per-node=128
#SBATCH --job-name test_mpi-%j
#SBATCH -o outage_mpi-%j.out
#SBATCH -e outage_mpi-%j.err

cd /global/cfs/cdirs/m4388/**Enter_your_Group3_Directory_here**/anatomy_of_a_power_outage/

module load conda
conda activate /global/cfs/cdirs/m4388/2025_BootCamp_ORNL/anatomy_of_a_power_outage/env/bootcamp

srun -n1 python GenerateOutageCSV.py 2014 2015 2016 2017 2018 2019 1.0
srun -n1 python GenerateOutageCSV.py 2020 2021 1.0
```

The comment lines beginning with ```SBATCH``` tell Slurm what machine configurations we want. The number of nodes, the requested amount of walltime, and the number of processes ran on each node are some examples of configurations we request. For this code, we will leave the number of nodes at 1. 

Further down we can see two srun commands. These are the commands that actually run our Python script. We split it into to two runs due to the large amount of data we are working with. The Python script (GenerateOutageCSV) takes multiple arguments, that are passed in here the batch script: a list of years to evaluate, and an outage length threshold which is a floating point number representing the minimum number of hours an outage lasts to qualify for recording in our final CSV.

Additionally, we use the ```-n``` option to specify the number of processes given to each job. Right now we are only giving 1 to each, but we will use this option to investigate speedup with MPI.

## To Do: Edit Your Slurm Script so you have the right path to your project and submit your first job. 

1. Open the Terminal:
   - Click the ➕ button at the top of this notebook to open the Launcher.
   - Scroll down to the **"Other"** section and click **Terminal**.

2. Open the Slurm script in the `vi` editor:
   ```bash
   vi submit_python_outage.sbatch
   ```

3. Enter edit mode:
   - Press the **`i`** key (stands for *insert*) to start editing.

4. Find the following line:
   ```bash
   cd /global/cfs/cdirs/m4388/**Enter_your_Group3_Directory_here**/anatomy_of_a_power_outage/
   ```
   Replace `**Enter_your_Group3_Directory_here**` with either `Group3a` or `Group3b`, depending on your assigned group.

5. Save and exit:
   - Press the **`Esc`** key to leave edit mode.
   - Type `:wq` and press **Enter** to write (save) and quit the editor.


Below is a cell which can submit our batch script to the queue.

In [ ]:
ID=!sbatch --parsable submit_python_outage.sbatch

If you want to check whether your job is running, go back to the terminal and enter the following command, replacing YOUR_UID with your username on Perlmutter:

```squeue -u YOUR_UID ```

## Tracking Speedup with MPI

You should have noticed two files generated in the directory where we are running:

```outage_csv-{ID}.out``` 

```outage_csv-{ID}.err```

 hese files contain the standard output and any errors produced during the submitted job. The `{ID}` will be replaced by the job number assigned by the scheduler to track your job.

Look in the ```.out``` file and you should see two lines recording how long each srun call.

# Part 1 of the Project, The Speedup Experiment 

Below is a cell with a hard-coded Python dictionary called `timings`. You will use this dictionary and the batch script to conduct the scaling study for the first part of your project.

In the dictionary:

- The **number on the left of the `:`** is the number of MPI processes used to run the `GenerateOutageCSV.py` script (e.g., 1, 2, 4, ..., 128).
- The **number on the right of the `:`** will be the time it takes to run the script with that number of processes.

### What to Do

1. **Run your batch script with 1 process** (using `-n 1`).
   - After the job completes, open the `outage_csv-{ID}.out` file.
   - Find the total processing time for each set of files there reported there.
   - Sum the two times together to get the total time. 
   - Record that time by replacing the value next to `1:` in the dictionary.

2. **Repeat this process for the other process counts** listed in the dictionary (e.g., `-n 2`, `-n 4`, ..., `-n 128`).
   - For each run, update the `-n` option in the batch script to the new number of processes.
   - After each run, open the `outage_csv-{ID}.out` file and compute and record the total time.
   - Replace the value in the dictionary for that process count with the new total timing.

3. **When you’ve completed all runs**, your dictionary will contain the measured runtimes for each number of processes.

You can then use the plotting cell to visualize how runtime changes with the number of MPI processes.


In [ ]:
timings = {1: 1,
           2: 1,
           4: 1,
           8: 1,
           16: 1,
           32: 1,
           64: 1,
           128: 1,
          }

Now let's plot the timings and see how increasing the number of processes improves (or degrades) our performance.

In [ ]:
plt.scatter(timings.keys(), timings.values())
plt.title("Speedup via MPI")
plt.xlabel("Number of Processes")
plt.ylabel("Total time (seconds)")

# Data Analysis

### The Big Question for Part 2 of Your Project

**Can you use this newly cleaned data to show that power outages have some characteristics similar to the characteristics of severe weather?**


First we will give you some functions to help you analyse the power outage data with the newly cleaned data. 

## Helpers

In the following section, you are provided several helper functions that will help aid you in your workflow and analysis.

Below is a brief explanation of these functions and what they can be used for:

- circular_toy
    - This function takes a time of year (toy) value from an outage entry, and calculates two components of that value using some basic trig functions. 
- plot_range_slice
    - This function takes your outage data, the number of clusters you're separating it into, and a lower and upper range of time of year and plots it into a 2d slice of the data from that timeframe. You can optionally choose to scale the points by how many people are affected by each outage.
- revert_comp
    - This function takes 2 components of a time of year and converts them back to their original value.
- centroid toy
    - This function takes the centroids of all clusters and the scaler used to normalize the original data and converts the centroids to a time of year value for each cluster.
- concat helper
    - This function takes two file names, each containing a csv of outages, and concatenates them into a single csv file.
 
TODO : "Shift" + "return" or "enter" in each of these cells to activiate them.  

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
def filter_column(df, column, filter_val):
    mask = (df[column] == filter_val)
    return df[mask]

In [ ]:
def circular_toy(toy):

    x = (2*math.pi*(toy-1)) / 12
    toy_sin = math.sin(x)
    toy_cos = math.cos(x)
    
    return toy_sin, toy_cos

In [ ]:
def plot_range_slice(data, k, lower, upper, scale=False):
    low_sin, low_cos = circular_toy(lower)
    up_sin, up_cos = circular_toy(upper)
    # print(f'Low Sin: {low_sin}, Low Cos: {low_cos}, High Sin: {up_sin}, High Cos: {up_cos}')
    # date_mask = (new_data['Month'] >= lower) & (new_data['Month'] < upper)
    date_mask = None
    if lower < upper:
        date_mask = (data['Month'] >= lower) & (data['Month'] < upper)
    else:
        date_mask = (data['Month'] >= lower) | (data['Month'] < upper)
    masked_data = data[date_mask]
    # print(masked_data)
    fig = plt.figure(figsize=(12, 10))
    
    ax = fig.add_subplot()
    for i in range(k):
        cluster_data = masked_data[masked_data['clusters'] == i]
        if scale:
            sizes = cluster_data['Sum']/30
        else:
            sizes = 100
        ax.set_aspect('equal', adjustable='box')
        ax.scatter(cluster_data['Long'], cluster_data['Lat'], label=f'Cluster {i+1}', edgecolor='black', alpha=0.5, s=sizes)
    
    ax.set_title(f'Clusters of Outages for Time Between Month {lower} and {upper}')
    lgnd = ax.legend(markerscale=1)
    for handle in lgnd.legend_handles:
        handle.set_sizes([24.0])

In [ ]:
def revert_comp(sin, cos):
    theta = math.atan2(sin, cos)

    x = 1 + (12 * theta) / (2 * math.pi)

    if x < 1:
        x += 12
    
    return x

In [ ]:
def centroid_toy(centroids, scaler):
    inversed_centroids = scaler.inverse_transform(centroids)
    for i in range(0, len(inversed_centroids)):
        print(f"Centroid for Cluster {i+1} occurs during time of year: {revert_comp(inversed_centroids[i][2], inversed_centroids[i][3])}")

In [ ]:
def concatHelper(filename1, filename2):
    # Read the two CSV files into dataframes
    first = pd.read_csv(f"Data/OutageCSVs/{filename1}")
    second = pd.read_csv(f"Data/OutageCSVs/{filename2}")
    
    # Concatenate the two dataframes
    df_concat = pd.concat([first, second], ignore_index=True)
    
    # Save the concatenated dataframe to a new CSV file (optional)
    df_concat.to_csv('Data/OutageCSVs/AllOutages.csv', index=False)

In [ ]:
def get_data(filename="AllOutages"):
    data = pd.read_csv(f"Data/OutageCSVs/{filename}.csv")

    data['year'] = pd.to_datetime(data['OutageStart'])
    data['year'] = data['year'].dt.year

    drop_indices = data[data['Long']<=-130].index
    data.drop(drop_indices, inplace=True)

    return data

In [ ]:
concatHelper("2014-2019-Outages.csv", "2020-2021-Outages.csv")

# Exploring the data

Let's read in the data and look at it. We will use the ```get_data``` helper function from above.
Then we'll use the python print funciton to view it. 



In [ ]:
data = get_data()
print(data.head(200))

Here is what the column headings mean:

- **State**: The name of the state where the outage was located.  
- **FIPS**: The code that corresponds to the state and county where the outage occurred.  
- **StateNum**: The state number extracted from the FIPS code.  
- **Region**: The larger U.S. region where the outage was located. See the map at the top of this notebook for regional definitions.  
- **Lat**: Latitude of the outage location.  
- **Long**: Longitude of the outage location.  
- **Month**: The month during which the outage took place (across any year in the dataset).  
- **Month_Sin**: The date expressed as a cyclic component that helps ensure outages occurring at the end of one year (e.g., December) and at the beginning of the next (e.g., January) are grouped together.
- **Month_Cos**: The date expressed as a second cyclic component used similarly to keep outages at the end and beginning of the year together.
- **OutageStart**: The date and time when the outage begins.
- **OutageEnd**: The date and time when the outage ends.
- **OutageLength**: The duration of each outage, calculated for all customers in each county.

> **Note:** The data cleaning script collected power outage data for every county in the US at 15-minute intervals for all dates between 2016 and 2022. It defines an outage start as the first date and time when more than 10% of the total customers in a county are without power, and the outage length is calculated until the total number of customers with power falls below the 10% threshold.

- **Sum**: Represents the average number of customers who were without power over the duration of each outage for each county. 
- **Year**: The year in which the outage occurred.


As you can see, this dataset gives you many different opportunities to sort or aggregate the data.

For example you could look at the average nubmer of outages per state per month. And plot histograms of each region. 

### Grouping by State and Month to Find Average Number of Outages

To compute the average number of outages per state per month, follow these steps:

1. Group the data by `State` and `Month`
2. Count the number of outages in each group
3. Compute the average number of outages per month per state

Here's how to do it in code:

In [ ]:
# Aggregate the data by Average nubmer of outages per state. 
#groupby is a function in the pandas Python package that groups data by one or more keys, allowing you to perform aggregate operations on each group separately.
#We use it below to group data by State. 

# Make sure OutageStart is in datetime format
data['OutageStart'] = pd.to_datetime(data['OutageStart'])

# Extract month and year as separate columns if not already present
data['MonthInt'] = data['OutageStart'].dt.month
data['Year'] = data['OutageStart'].dt.year

# Count number of outages per State, Year, and Month
# Uses .size() to count how many records (rows) are in each group — i.e., how many outages happened in each state-month-year combination.
monthly_counts = data.groupby(['State', 'Year', 'MonthInt']).size().reset_index(name='OutageCount')

# Computes the average of the OutageCount for each (State, Month) pair over multiple years.
average_outages = monthly_counts.groupby(['State', 'MonthInt'])['OutageCount'].mean().reset_index(name='AvgOutages')

# View the result
print(average_outages)

Now if you wanted to plot a histogram for the sate of texas to show wich months had the largest number of outages. you could do something like this: 

In [ ]:
import matplotlib.pyplot as plt

# Filter the data for Texas
texas_avg = average_outages[average_outages['State'] == 'Texas']
texas_avg


# Plot a bar chart of average outages per month
plt.figure(figsize=(10, 6))
plt.bar(texas_avg['MonthInt'], texas_avg['AvgOutages'], color='skyblue', edgecolor='black')
plt.title('Average Number of Outages per Month in Texas')
plt.xlabel('Month')
plt.ylabel('Average Number of Outages')
plt.xticks(range(1, 13))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
texas_avg

If you want to rewrite the code to find the average number of outages per region rather than per state, you’ll need to make a few key changes:

1. Update the groupby clause in the code above
 Replace "State" with "Region" everywhere inside the groupby function.

2. Update the filtering line in the plotting code

Replace:
```texas_avg = average_outages[average_outages['State'] == 'Texas']```

With:

```region_avg = average_outages[average_outages['Region'] == 4]```

**Note:** We removed the quotes from 4 so that Python reads it as an integer. If we had kept the quotes ('4'), Python would interpret it as a string, and the plot would not work correctly.

3. Fix all axis labels and titles

Anywhere your plot labels mention "State" or "Texas", update it to reflect "Region" and the region number you’re plotting.

**Why This Matters**  
We want to see if the outages follow the same cyclic patterns as the weather. Looking at the average number of outages per region per month gives a rough estimate of the monthly trends in the outage data.

Also, making a histogram like this for each region is a great sanity check before moving into the machine learning phase of your project, which will use a different method to determine when and where clusters of outages are forming.

# Part 2a of the Project: Sanity Check with Histograms  
Designate a team in your group to copy and adjust the code above so it creates a histogram of the average number of outages per region per month — for all regions.

You’ll use these plots as sanity checks in the next part of the project, so make sure each team documents:
* Which months in each region have the most outages on average over all the data.
* What changes they made to the code cells to get the plots.

Save the plots in an organized way in case you want to use any of them for your final presentation.


In [ ]:
# TODO: Aggregate the data by average number of outages per state.

In [ ]:
# TODO: Make histogram plots of the average outages per month per region.
# Save the plots for later reference, and make sure the filenames indicate the state and region shown.

# Part 2b of the Project: Unsupervised Learning (KMeans Clustering)

### Unsupervised Learning and K-Means Clustering

Unsupervised learning refers to machine learning techniques that categorize data without using initial labels. One common technique is **K-Means Clustering**.

K-Means requires the user to specify the number of clusters. It begins by selecting initial cluster centroids (i.e., points that represent the center of each cluster). Each data point is then assigned to the nearest centroid, typically using Euclidean distance. The centroids are recalculated as the mean of the points in each cluster. This process repeats until the centroids converge (i.e., change very little between iterations).

📖 More about Scikit-Learn’s K-Means:  
https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

---

### Your Task

In the next part of your project, you will use K-Means to cluster power outage data based on **latitude, longitude, and time of year**. 

The code blocks below include several tests to help you choose the number of clusters that results in the best overall fit. 
These tests aim to help you evaluate cluster quality and centroid separation.


## Additional Imports
Please click on the cell below and press **Shift + Enter** (or **Return**) to run it and import the Scikit-learn packages.

---

### What is Scikit-learn?

**Scikit-learn** is a popular Python library for machine learning. It provides simple and efficient tools for tasks like classification, regression, clustering, dimensionality reduction, and model evaluation.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

## Methods of Detemining K

As previously stated, K-Means requires the number of clusters desired (k) to be provided. Knowing this, how will we decide on the correct number of clusters to sort our data into? There are several methods of picking the best k. Below we will discuss three of these techniques.

### Elbow Method

The elbow method is one such method to determine how many clusters (k) is ideal for your data. Typically, a certain range of k values is tested and for each k value we perform K-Means. Then, for each k, we find the within cluster sum of squares (WCSS) for each cluster found for that k. We can plot each WCSS value against its respective k value and determine the suggested number of categories by pinpointing where the addition of more clusters minimally decreases the WCSS. This is known as the elbow point and is located at the recommended number of clusters.

In [ ]:
def elbow_method(data, columns=['Region', 'Month', 'OutageLength'], year=None):
    if year:
        drop_indices = data[data['year'] != year].index
        data.drop(drop_indices, inplace=True)
    
    # drop_indices = data[data['State']=="Texas"].index
    # data.drop(drop_indices, inplace=True)
    
    X = data[columns]
    X=X.dropna()

    scaler = StandardScaler()
    norm_X = scaler.fit_transform(X)

    wcss = []
    for i in range(1, 30):
       kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=0)
       kmeans.fit(norm_X)
       wcss.append(kmeans.inertia_)
    plt.plot(range(1,30), wcss)
    plt.title('Elbow Method')
    plt.xlabel('Number of Clusters')
    plt.ylabel('WCSS')
    # plt.savefig(f"Plots/elbow-{filename}.png")
    plt.show()

    return norm_X, scaler

In [ ]:
data = get_data()
data

### Running the K-Means Elbow Method

The cell below calls the `elbow_method` function defined above. Here, we use the `'Month_Sin'` and `'Month_Cos'` components instead of raw month values to account for the cyclical nature of time and smooth transitions across years.

The plot it produces shows **WSSC** (within-cluster sum of squares cost) as a function of the number of clusters. You’re looking for the **"elbow"**—the point on the X-axis where the WSSC begins to level off. 

This "elbow point" is a good estimate of the optimal number of clusters. It represents the point beyond which adding more clusters yields diminishing returns in reducing the average distance between centroids and their associated points.

In [ ]:
norm_X, scaler = elbow_method(data, columns=['Long', 'Lat', 'Month_Sin', 'Month_Cos'])

## Silhouette Method

On some occasions, the elbow method will not have an easily observable elbow point. Thus, it may be neccessary to try other methods of determining k. One such method is calculating the Silhouette Score for different values of k. Similar to the Elbow Method, we begin by iterating through different values of k. For each one, we let K-Means determine the groupings for that amount of cluster then calculate a Silhouette Score (which should be between -1 and 1) for that k. The k value with the __highest__ Silhouette Score (closest to 1) is the recommended number of clusters for your data.

In [ ]:
from sklearn.metrics import silhouette_score

for k in range(2, 10):
    clusterer = KMeans(n_clusters=k, random_state=10,n_init='auto')
    labels = clusterer.fit_predict(norm_X)

    print(f"{k} clusters the Silhouette Score is: {silhouette_score(norm_X, labels)}")

## Davies-Bouldin Index

Another method we can use that provides a more explicit numerical value over elbow method is the Davies-Bouldin Index. The Davies-Bouldin index measures similarity between clusters and their most similar cluster by taking the ratio of within cluster distances to between cluster distances. This value indicates a better k when the Davies-Bouldin score is __lower__

SKLearn's Davies-Bouldin Index: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.davies_bouldin_score.html#sklearn.metrics.davies_bouldin_score

In [ ]:
from sklearn.metrics import davies_bouldin_score
for k in range(2,20):
    clusterer = KMeans(n_clusters=k, random_state=10,n_init='auto')
    labels = clusterer.fit_predict(norm_X)

    print(f"{k} clusters the Davies-Bouldin Score is: {davies_bouldin_score(norm_X, labels)}")

## Calinski-Harabasz Index

The final method to determine the best k we will be looking at is the Calinsko-Harabasz Index. This score evaluates the ratio of the sums of between cluster dispersion and within cluster dispersion. It is also known as the Variance Ratio Criterion. Here we are looking for the __highest__ value to signal what our best k is. 

SKLearn's Calinski-Harabasz Index: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.calinski_harabasz_score.html#sklearn.metrics.calinski_harabasz_score

In [ ]:
from sklearn.metrics import calinski_harabasz_score
for k in range(2,20):
    clusterer = KMeans(n_clusters=k, random_state=10,n_init='auto')
    labels = clusterer.fit_predict(norm_X)

    print(f"{k} clusters the Calinski-Harabasz Score is: {calinski_harabasz_score(norm_X, labels)}")

## K-Means

After running through the different methods of determining k above, you should have some good ideas about what k(s) should be best. Now you can test them and see what the clustering looks like when tranlated to a graph. Below is a clustering function that plots our data with color-coded clusters based on K-Means. Try playing around with the value of k or even the columns to plot by (make sure to plot 2 or 3 columns only) to see what patterns you can find. (Note that changing the columns only changes how the points are plotted and does not determine what features are used to cluster them as that was decided when we created ```norm_X``` in the elbow method)

In [ ]:
def cluster(data, norm_X, k=3, columns=['Region', 'Month', 'OutageLength'], scale=False):
    dims = len(columns)
    if dims > 3 or dims < 2:
        print("Should be looking at 2 or 3 features, change number of columns evaluated.")

    kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=0)
    data['clusters'] = kmeans.fit_predict(norm_X)

    fig = plt.figure(figsize=(16, 14))

    # 3d clustering
    if dims == 3:
        ax = fig.add_subplot(111, projection='3d')
    # 2d clustering
    else:
        ax = fig.add_subplot()

    for i in range(k):
        cluster_data = data[data['clusters'] == i]
        if scale:
            sizes = cluster_data['Sum']
        else:
            sizes = 10
        if dims == 3:
            ax.scatter(cluster_data[columns[0]], cluster_data[columns[1]], cluster_data[columns[2]], label=f'Cluster {i + 1}', edgecolor='black', alpha=0.5, s=sizes/30)
        else:
            ax.scatter(cluster_data[columns[0]], cluster_data[columns[1]], label=f'Cluster {i + 1}', edgecolor='black', alpha=0.5, s=sizes/30)

    ax.set_title('Clusters of Outages')
    ax.set_xlabel(columns[0])
    ax.set_ylabel(columns[1])
    if dims == 3:
        ax.set_zlabel(columns[2])
    lgnd = ax.legend(markerscale=1)
    for handle in lgnd.legend_handles:
        handle.set_sizes([24.0])
    return data, kmeans.cluster_centers_

In [ ]:
k = 7

In [ ]:
# range_mask = (data['Month'] >= 6.0) & (data['Month'] < 12.0)
new_data, centroids = cluster(data, norm_X, k, columns=['Long', 'Lat', 'Month'], scale=True)

We can use one of our helper functions, ```centroid_toy```, to find the time of year where each cluster is centered at. 

In [ ]:
centroid_toy(centroids, scaler)

Another one of our helper functions, ```plot_range_slice```, can be used to look at our data in a 2D slice during a range of time we specify. We can use this to evaluate clusters in different regions during times of the year that we are interested in. (Note that you can specify ranges where the lower bound is a month following the upper bound and the range will circle back around the year (i.e. lower = 12 upper = 2 will show data from December to February)  

In [ ]:
plot_range_slice(data, k, 1.0, 12.0, scale=True)

Here we use the ```plot_range_slice``` function to create graphs of each month to see how our data changes and is clustered across the year.

In [ ]:
for i in range(1, 13):
    plot_range_slice(new_data, 7, i, i + 1, scale=True)

### Step 1: Refine your clusters
1. Test and adjust the number of power outage clusters based on the results and tests in the notebook.

### Step 2: Analyze and characterize the results
- This is the process of determining what story the data and results tell you.

Questions to consider:
- When are the centers of those clusters for each region?
- What are the general characteristics of the power outages in those clusters?
  - Are they generally long or short outages?
  - How many customers are impacted on average?
- Are there regions or states that mostly share clusters or that mostly do not share clusters?
  - Why might that be in each case?
  - Form some hypotheses about the possible relationships to:
    - Regional climate and weather
    - The time of year
    - The severity of the power outages
- How well do you trust the cluster results?

### Step 3: Consider alternative ways to characterize the data
1. Are there other ways besides K-means to support or challenge the cluster centers?
   - Try histograms or other plots
   - Consider other machine learning or statistical methods
2. If you don’t have time to try another method:
   - Look for anecdotal support
   - Example: link your clusters to well-known power outage events found via news search

---

# Go to the Weather.ipynb to do part 3 of your projct. 